# ECC/EMC — Kaggle Training: EmoSet (Single Alpha)

Supervised training on EmoSet-118K, one alpha per run.

## Before you run
1. **Enable GPU:** Settings → Session options → GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Public EmoSet-118K dataset in the sidebar.
3. **Persistent checkpoints:** Add `your-username/ecc-checkpoints` dataset and set `PERSISTENT_DIR`.
4. **GitHub token secret:** `GITHUB_TOKEN` in Kaggle Settings → Secrets.
5. **Edit `ALPHA`** below (0.0, 0.2, or 1.0).

## Why one alpha per run?
Each alpha takes ~10–12 hours on P100. One alpha per Kaggle session (12-hour cap).

## Results
Outputs go to `/kaggle/working/runs/emoset_r50_a<alpha>/`.
A zip cell at the end creates `emoset_results.zip` for one-click download.


In [ ]:
# ===================== CONFIGURE THESE =====================
# Alpha: 0.0 (CE), 0.2 (ListMLE), or 1.0 (ListMLE)
ALPHA = 0.2   # <-- CHANGE THIS for each run

# Public EmoSet-118K dataset
EMOSET_DATASET_SLUG = "georgeliil/emoset"   # <-- REPLACE if different

# Persistent checkpoint Dataset (recommended)
PERSISTENT_DIR = ""   # e.g. "/kaggle/input/your-username/ecc-checkpoints"

# ==========================================================

print("Alpha:", ALPHA)
print("EmoSet dataset slug:", EMOSET_DATASET_SLUG)
print("Persistent dir:", PERSISTENT_DIR or "(disabled)")

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone repo using GitHub token secret (added in Kaggle Settings → Secrets)
import os
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{secret_token}@github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"
except Exception:
    repo_url = "https://github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"

dest = "/kaggle/working/ecc-ver"

if not os.path.exists(dest):
    subprocess.run(["git", "clone", repo_url, dest], check=True)
    print("Cloned repo to", dest)
else:
    print("Repository already exists at", dest)

os.chdir(dest)
print("Current dir:", os.getcwd())

In [ ]:
# Install ecc-ver package
!pip -q install -e .

In [ ]:
# Symlink EmoSet dataset into data/ layout
import os
from pathlib import Path

os.makedirs("data", exist_ok=True)

# Robustly find EmoSet/ anywhere under /kaggle/input
# Kaggle mounts private datasets under /kaggle/input/datasets/<user>/<slug>/...
found_images = False
found_splits = False
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 5:
        dirs[:] = []
        continue
    for d in dirs:
        if d in ("image", "EmoSet") and not found_images:
            src = os.path.join(root, d)
            if any((Path(src) / c).is_dir() for c in ["amusement", "anger", "awe", "contentment", "disgust", "excitement", "fear", "sadness"]):
                dst = "data/EmoSet"
                if not os.path.exists(dst):
                    os.symlink(src, dst)
                    print(f"Linked EmoSet images -> data/EmoSet (from {src})")
                found_images = True
        if d == "EmoSet_splits" and not found_splits:
            src = os.path.join(root, d)
            dst = "data/EmoSet_splits"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked EmoSet splits -> data/EmoSet_splits (from {src})")
            found_splits = True

if not found_images:
    print("WARNING: Could not find EmoSet image directory under /kaggle/input")
if not found_splits:
    print("WARNING: Could not find EmoSet_splits under /kaggle/input")

print("\nData layout:")
!find data -maxdepth 3 | head -50

In [ ]:
# Quick smoke test
!python -m pytest -q

In [ ]:
# ===================== SUPERVISED (EmoSet, ResNet50) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

persistent_arg = f"--set persistent_dir={PERSISTENT_DIR}" if PERSISTENT_DIR else ""

print(f"=== alpha={ALPHA} ===")
cmd = f"python -m ecc.supervised --config configs/emoset_resnet50.yaml --set loss.alpha={ALPHA} {persistent_arg} output_dir=/kaggle/working/runs/emoset_r50_a{ALPHA}"
print("Running:", cmd)
!{cmd}

print("Supervised done.")

In [ ]:
# ===================== PREPARE RESULTS ZIP FOR ONE-CLICK DOWNLOAD =====================
import zipfile, os, glob, shutil

runs_dir = "/kaggle/working/runs"
zip_path = f"/kaggle/working/emoset_r50_a{ALPHA}_results.zip"

run_dir = os.path.join(runs_dir, f"emoset_r50_a{ALPHA}")

if not os.path.exists(run_dir):
    print(f"No run directory found: {run_dir}")
else:
    files_to_zip = ["best.pt", "checkpoint.pt", "results.json", "history.jsonl", "config.json"]
    
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in files_to_zip:
            fpath = os.path.join(run_dir, fname)
            if os.path.exists(fpath):
                zf.write(fpath, arcname=fname)
                print(f"Added {fname}")
            else:
                print(f"Skipped {fname} (not found)")
    
    print(f"\nCreated {zip_path}")
    print("Contents:")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            print(f"  {info.filename} ({info.file_size:,} bytes)")
    
    print("\nDownload from Output panel (right sidebar → Output).")

## Resuming after session timeout

Kaggle free sessions are capped at ~12 hours. Every epoch saves `checkpoint.pt` to:
1. The run's `output_dir` (`/kaggle/working/runs/emoset_r50_a<alpha>/`), and
2. `PERSISTENT_DIR` (your mounted Kaggle Dataset, if set).

If a session dies, re-run the same cell — it auto-detects `checkpoint.pt` in the output dir
**and** in `PERSISTENT_DIR`, resuming from the last epoch.

To force a specific checkpoint, add `--set resume_from=/kaggle/working/runs/emoset_r50_a<alpha>/checkpoint.pt`.

## Download results

Run the zip cell above to create `emoset_r50_a<alpha>_results.zip` containing:
- `best.pt` — best-on-val model weights
- `checkpoint.pt` — resumable training checkpoint
- `results.json` — test metrics (ACC, ECC, EMC, ACC2)
- `history.jsonl` — per-epoch training log
- `config.json` — exact config used

Download the zip from the notebook output panel before the session ends.

## For the next alpha

Change `ALPHA` at the top of this notebook and run again. Each alpha is a separate Kaggle session.
